# Step 9 Hybrid Fusion

This notebook builds the final submission model by blending the Step 7 ensemble with the best Step 8 text-only model.
The logic stays light on engineering and heavy on clean OOF validation.


## Environment Check

The helper cell also keeps the source campaigns easy to trace.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

EXPECTED_ENV = 'usc-nlp'
if EXPECTED_ENV not in sys.executable.lower():
    raise RuntimeError(f'This notebook should run inside the {EXPECTED_ENV} conda environment. Current executable: {sys.executable}')


def detect_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'group17_runtime.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root from the notebook location.')


def find_latest_run(root: Path, prefix: str) -> Path:
    candidates = [path for path in root.iterdir() if path.is_dir() and path.name.startswith(prefix)]
    if not candidates:
        raise FileNotFoundError(f'No run found in {root} with prefix {prefix!r}.')
    return max(candidates, key=lambda path: path.stat().st_mtime)


def read_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def run_command(cmd: list[str]) -> None:
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)


PROJECT_DIR = detect_project_dir(Path.cwd().resolve())
print(f'Project directory: {PROJECT_DIR}')
print(f'Python executable: {sys.executable}')


Project directory: D:\CS544-Group17-Project
Python executable: f:\anaconda\envs\usc-nlp\python.exe


## Source Campaigns

Step 9 reads the newest submission-style Step 7 and Step 8 runs.


In [2]:
STEP7_RUN_ID = 'exp_s71_screen_fusion_submission'
STEP8_RUN_ID = 'exp_s81_textonly_refresh_submission'
STEP9_SCRIPT = PROJECT_DIR / 'group17_step9_hybrid_fusion.py'
STEP7_ROOT = PROJECT_DIR / 'step7_outputs'
STEP8_ROOT = PROJECT_DIR / 'step8_outputs'
OUTPUT_ROOT = PROJECT_DIR / 'step9_outputs'
RUN_ID = 'exp_s91_hybrid_fusion_submission'
FUSION_ID = 's91_step7_step8_hybrid_blend_v1'

STEP7_RUN_DIR = find_latest_run(STEP7_ROOT, STEP7_RUN_ID)
STEP8_RUN_DIR = find_latest_run(STEP8_ROOT, STEP8_RUN_ID)
print(f'Source Step 7 campaign: {STEP7_RUN_DIR}')
print(f'Source Step 8 campaign: {STEP8_RUN_DIR}')


Source Step 7 campaign: D:\CS544-Group17-Project\step7_outputs\exp_s71_screen_fusion_submission_local_20260424_083551
Source Step 8 campaign: D:\CS544-Group17-Project\step8_outputs\exp_s81_textonly_refresh_submission_local_20260424_084702


## Launch Step 9

The command pins both source campaigns so the final blend stays reproducible.


In [3]:
step9_cmd = [
    sys.executable,
    str(STEP9_SCRIPT),
    '--primary-campaign-dir', str(STEP7_RUN_DIR),
    '--secondary-campaign-dir', str(STEP8_RUN_DIR),
    '--run-id', RUN_ID,
    '--fusion-id', FUSION_ID,
]
run_command(step9_cmd)
STEP9_RUN_DIR = find_latest_run(OUTPUT_ROOT, RUN_ID)
print(f'Step 9 run directory: {STEP9_RUN_DIR}')


Running command:
f:\anaconda\envs\usc-nlp\python.exe D:\CS544-Group17-Project\group17_step9_hybrid_fusion.py --primary-campaign-dir D:\CS544-Group17-Project\step7_outputs\exp_s71_screen_fusion_submission_local_20260424_083551 --secondary-campaign-dir D:\CS544-Group17-Project\step8_outputs\exp_s81_textonly_refresh_submission_local_20260424_084702 --run-id exp_s91_hybrid_fusion_submission --fusion-id s91_step7_step8_hybrid_blend_v1
Step 9 run directory: D:\CS544-Group17-Project\step9_outputs\exp_s91_hybrid_fusion_submission


## Weight Search

The top rows show where the best hybrid blend landed.


In [4]:
weight_path = STEP9_RUN_DIR / FUSION_ID / f'{FUSION_ID}_weight_search.csv'
weight_search = pd.read_csv(weight_path)
display(weight_search.head(10))


,primary_id,secondary_id,primary_weight,secondary_weight,default_F1,best_threshold,tuned_F1,tuned_Precision,tuned_Recall,tuned_Accuracy
0,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.10,0.90,0.809060,0.71,0.812204,0.847175,0.780006,0.846595
1,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.11,0.89,0.808680,0.70,0.812327,0.846703,0.780633,0.846595
2,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.12,0.88,0.808490,0.70,0.812653,0.847043,0.780946,0.846861
3,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.13,0.87,0.808490,0.70,0.812653,0.847043,0.780946,0.846861
4,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.14,0.86,0.808748,0.72,0.812367,0.849760,0.778126,0.847128
5,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.15,0.85,0.808939,0.69,0.812459,0.846991,0.780633,0.846728
6,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.16,0.84,0.809068,0.71,0.812490,0.849282,0.778753,0.847128
7,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.17,0.83,0.809197,0.71,0.812490,0.849282,0.778753,0.847128
8,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.18,0.82,0.809197,0.71,0.812888,0.850154,0.778753,0.847528
9,s71_step6_screen_blend_v1,s84_bertweet_textonly_long_v1,0.19,0.81,0.809387,0.75,0.812840,0.856103,0.773739,0.848461


## Final Step 9 Metrics

This summary is the cleanest record of the final submission model.


In [5]:
summary_path = STEP9_RUN_DIR / FUSION_ID / f'{FUSION_ID}_summary.json'
step9_summary = read_json(summary_path)
final_step9 = pd.DataFrame([step9_summary['tuned_threshold']])
display(final_step9)
print(f"Primary weight: {step9_summary['primary_weight']}")
print(f"Secondary weight: {step9_summary['secondary_weight']}")
print(f"Best threshold: {step9_summary['best_threshold']}")


,F1,Precision,Recall,Accuracy
0,0.816929,0.846438,0.789408,0.849527


Primary weight: 0.72
Secondary weight: 0.28
Best threshold: 0.53


## Submission Preview

The final CSV lives in the Step 9 folder. This quick preview is enough to confirm the format before upload.


In [6]:
submission_path = STEP9_RUN_DIR / FUSION_ID / f'submission_{FUSION_ID}.csv'
submission_preview = pd.read_csv(submission_path)
print(f'Submission path: {submission_path}')
display(submission_preview.head())


Submission path: D:\CS544-Group17-Project\step9_outputs\exp_s91_hybrid_fusion_submission\s91_step7_step8_hybrid_blend_v1\submission_s91_step7_step8_hybrid_blend_v1.csv


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
